# VoiceForge — GPU backend

Runs Chatterbox + Qwen3-TTS on this Colab's free T4 GPU and exposes them as a public HTTPS API via a Cloudflare tunnel.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

Then **Runtime → Run all**. Cell 3 prints the backend URL — paste it into VoiceForge. Keep this tab open while generating; the URL is new every session.

In [ ]:
# Cell 1 — Install
!pip install -q chatterbox-tts fastapi uvicorn nest-asyncio python-multipart soundfile
!pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("Installed. Restart runtime if prompted, then run Cell 2.")

In [ ]:
# Cell 2 — Load models
import torch, gc
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0))

MODELS = {}

from chatterbox.tts import ChatterboxTTS
MODELS["chatterbox"] = ChatterboxTTS.from_pretrained(device=DEVICE)
print("chatterbox ready")

# Qwen3 is optional — comment out to save ~4 min of load time and VRAM.
try:
    from qwen3_tts import Qwen3TTSModel
    MODELS["qwen3"] = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        device_map=DEVICE, dtype=torch.float16,
        attn_implementation="sdpa",   # T4 does not support flash-attention-2
    )
    print("qwen3 ready")
except Exception as e:
    print("qwen3 unavailable:", e)

In [ ]:
# Cell 3 — API server + public URL
import io, os, uuid, base64, subprocess, threading, time, re
import numpy as np, soundfile as sf, torch, nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

VOICES = {}   # voice_id -> {"path": wav path, "transcript": str}
os.makedirs("/content/voices", exist_ok=True)

def wav_b64(audio, sr):
    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode()

@app.get("/health")
def health():
    return {
        "ok": True,
        "gpu": torch.cuda.get_device_name(0),
        "models": list(MODELS.keys()),
        "voices": list(VOICES.keys()),
    }

@app.post("/clone")
async def clone(audio: UploadFile = File(...), transcript: str = Form("")):
    vid = uuid.uuid4().hex[:12]
    path = f"/content/voices/{vid}.wav"
    data, sr = sf.read(io.BytesIO(await audio.read()))
    if data.ndim > 1:
        data = data.mean(axis=1)              # to mono
    sf.write(path, data, sr)
    VOICES[vid] = {"path": path, "transcript": transcript.strip()}
    return {"voice_id": vid, "duration": len(data) / sr}

@app.post("/generate")
async def generate(payload: dict):
    text  = payload["text"]
    vid   = payload["voice_id"]
    model = payload.get("model", "chatterbox")
    seed  = int(payload.get("seed", 0))
    if vid not in VOICES:
        return JSONResponse({"error": "unknown voice_id"}, status_code=400)
    if model not in MODELS:
        return JSONResponse({"error": f"{model} not loaded"}, status_code=400)

    torch.manual_seed(seed)
    ref = VOICES[vid]
    t0 = time.time()

    if model == "chatterbox":
        wav = MODELS["chatterbox"].generate(
            text,
            audio_prompt_path=ref["path"],
            exaggeration=float(payload.get("exaggeration", 0.4)),
            cfg_weight=float(payload.get("cfg", 0.5)),
            temperature=float(payload.get("temperature", 0.7)),
        )
        audio = wav.squeeze(0).cpu().numpy()
        sr = MODELS["chatterbox"].sr
    else:
        out = MODELS["qwen3"].generate(
            text=text,
            ref_audio=ref["path"],
            ref_text=ref["transcript"] or None,
            language="English",
            instruction=payload.get("style_prompt", ""),
        )
        audio, sr = out["audio"], out["sample_rate"]
        if torch.is_tensor(audio):
            audio = audio.squeeze().cpu().numpy()

    gc.collect(); torch.cuda.empty_cache()
    return {
        "audio_b64": wav_b64(audio, sr),
        "sample_rate": sr,
        "duration": len(audio) / sr,
        "gen_seconds": round(time.time() - t0, 2),
    }

# --- start server + tunnel ---
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error"),
    daemon=True,
).start()
time.sleep(3)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "=" * 60)
        print("  BACKEND URL — paste this into VoiceForge:")
        print("  " + m.group(0))
        print("=" * 60 + "\n")
        print("Keep this tab open while generating.")
        break

In [ ]:
# Cell 4 — Keep-alive (optional)
# Colab disconnects idle tabs. Run this in the background during long jobs.
import time
while True:
    time.sleep(60)
    print(".", end="", flush=True)